In [ ]:
import pandas as pd
import numpy as np
from keras import layers, Model
from keras.models import Sequential
from keras.optimizers import Adam
from keras.losses import MeanSquaredError
# import metric

# Загрузка данных
train_origin_path = "/content/train.zip/origin"
train_synth_path = "/content/train.zip/synth"
public_test_path = "/content/public_test.zip"


In [ ]:
!unzip public_test.zip





Archive:  public_test.zip
  inflating: 101.csv                 
  inflating: 110.csv                 
  inflating: 142.csv                 
  inflating: 164.csv                 
  inflating: 184.csv                 
  inflating: 20.csv                  
  inflating: 205.csv                 
  inflating: 207.csv                 
  inflating: 23.csv                  
  inflating: 238.csv                 
  inflating: 244.csv                 
  inflating: 266.csv                 
  inflating: 273.csv                 
  inflating: 284.csv                 
  inflating: 318.csv                 
  inflating: 328.csv                 
  inflating: 347.csv                 
  inflating: 358.csv                 
  inflating: 37.csv                  
  inflating: 374.csv                 
  inflating: 382.csv                 
  inflating: 406.csv                 
  inflating: 417.csv                 
  inflating: 425.csv                 
  inflating: 428.csv                 
  inflating: 436.csv    

In [ ]:
!unzip train.zip

In [ ]:


# Определение модели автоэнкодера
def build_autoencoder(input_dim):
    encoder = Sequential([
        layers.Dense(256, activation='relu', input_shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu')
    ])

    decoder = Sequential([
        layers.Dense(128, activation='relu', input_shape=(64,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(input_dim, activation='sigmoid')
    ])

    autoencoder = Model(inputs=encoder.input, outputs=decoder(encoder.output))
    autoencoder.compile(optimizer=Adam(), loss=MeanSquaredError())

    return autoencoder







In [ ]:
# Обучение модели на обучающей выборке
def train_model(train_origin_path, train_synth_path):
    origin_data = []
    synth_data = []

    # Считывание данных из обучающей выборки
    for filename in os.listdir(train_origin_path):
        if filename.endswith(".csv"):
            origin_data.append(pd.read_csv(os.path.join(train_origin_path, filename)))
    for filename in os.listdir(train_synth_path):
        if filename.endswith(".csv"):
            synth_data.append(pd.read_csv(os.path.join(train_synth_path, filename)))

    # Объединение данных в один датафрейм
    origin_data = pd.concat(origin_data, axis=0)
    synth_data = pd.concat(synth_data, axis=0)

    # Определение размеров входных данных
    input_dim = origin_data.shape[1]

    # Создание и обучение автоэнкодера
    autoencoder = build_autoencoder(input_dim)
    autoencoder.fit(synth_data, origin_data, epochs=10, batch_size=32)

    return autoencoder

In [ ]:
# Восстановление оригинальных данных из синтетических данных тестовой выборки
def predict_data(model, public_test_path):
    predictions = []

    # Считывание данных из тестовой выборки
    for filename in os.listdir(public_test_path):
        if filename.endswith(".csv"):
            synth_df = pd.read_csv(os.path.join(public_test_path, filename))
            predicted_df = pd.DataFrame(model.predict(synth_df), columns=synth_df.columns)
            predictions.append(predicted_df)

    return predictions

In [ ]:
# Расчет метрики DCR
def calculate_dcr(predictions, public_test_path):
    dcr_scores = []

    for i, filename in enumerate(os.listdir(public_test_path)):
        if filename.endswith(".csv"):
            original_df = pd.read_csv(os.path.join(public_test_path, filename))
            dcr_scores.append(metric.dcr(predictions[i], original_df))

    return np.mean(dcr_scores)

# Основной код
if __name__ == "__main__":
    # Убедитесь, что файлы train.zip и metric.py распакованы

    # Обучение модели
    train_origin_path = "train/origin"
    train_synth_path = "train/synth"
    autoencoder = train_model(train_origin_path, train_synth_path)

    # Восстановление оригинальных данных из синтетических
    public_test_path = "public_test"
    predictions = predict_data(autoencoder, public_test_path)

    # Расчет метрики DCR
    dcr_score = calculate_dcr(predictions, public_test_path)
    print("DCR:", dcr_score)


FileNotFoundError: [Errno 2] No such file or directory: 'train/origin'

In [ ]:
import pandas as pd
import zipfile
import os

# ... (ваш код для обучения модели и предсказания данных)

# Создание списка файлов для записи в ZIP-архив
files_to_zip = []
for i, prediction_df in enumerate(predictions):
    # Создайте уникальное имя для каждого файла
    filename = f"prediction_{i}.csv"
    prediction_df.to_csv(filename, index=False)
    files_to_zip.append(filename)

# Создание ZIP-архива
with zipfile.ZipFile('sample_submission_seed_0.zip', 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file)

# Удаление временных файлов .csv
for file in files_to_zip:
    os.remove(file)

print("Предсказания успешно записаны в sample_submission_seed_0.zip")


**5.02**

In [ ]:
df_train = pd.read_csv('startup_train.csv')
df_test = pd.read_csv('startup_test.csv')


In [ ]:
df_train["overview"] = df_train["overview"].fillna("")
df_train["tag_list"] = df_train["tag_list"].fillna("")

df_test["overview"] = df_test["overview"].fillna("")
df_test["tag_list"] = df_test["tag_list"].fillna("")

In [ ]:
numerical_features = df_train.select_dtypes(include=['float64', 'int64']).columns
text_features = ["overview"]
numerical_features = list(numerical_features)
numerical_features.remove('has_next_round')
y_train_df = df_train["has_next_round"]

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


# Создание векторов с использованием TF-IDF для "overview"
tfidf_overview = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_train_text = tfidf_overview.fit_transform(df_train["overview"])
X_test_text = tfidf_overview.transform(df_test["overview"])

# # Создание векторов с использованием TF-IDF для "tag_list"
# tfidf_tag = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
# X_train_category = tfidf_tag.fit_transform(df_train["tag_list"])
# X_test_category = tfidf_tag.transform(df_test["tag_list"])
df_train['tag_list_count'] = df_train.tag_list.str.count(',') + 1
df_train['tag_list_count'][df_train.tag_list.isna()] = 0
df_train['tag_list_no_count'] = df_train['tag_list'].isna()

df_test['tag_list_count'] = df_test.tag_list.str.count(',') + 1
df_test['tag_list_count'][df_test.tag_list.isna()] = 0
df_test['tag_list_no_count'] = df_test['tag_list'].isna()


In [ ]:
feature_names_text = [f"text_{i}" for i in range(X_train_text.shape[1])]
# feature_names_category = [f"category_{i}" for i in range(X_train_category.shape[1])]

X_train_text = pd.DataFrame(X_train_text.toarray(), columns=feature_names_text)
X_test_text = pd.DataFrame(X_test_text.toarray(), columns=feature_names_text)

In [ ]:
X_train_combined = pd.concat([df_train[numerical_features], X_train_text], axis=1)
X_test_combined = pd.concat([df_test[numerical_features], X_test_text], axis=1)

In [ ]:
columns_to_remove = ['index','has_raised_amount','ipo_prob']
# Удаляем столбцы из X_train_combined
X_train_combined = X_train_combined.drop(columns=columns_to_remove)

In [ ]:
columns_to_remove = ['index', 'has_raised_amount','ipo_prob']

# Удаляем столбцы из X_train_combined
X_test_combined = X_test_combined.drop(columns=columns_to_remove)

In [ ]:
from catboost import CatBoostClassifier
catboost = CatBoostClassifier(learning_rate = 0.03, l2_leaf_reg = 3, iterations = 300, depth = 6, silent = True)
catboost.fit(X_train_combined, y_train_df)

In [ ]:
# Предсказание для тестовых данных
y_pred = catboost.predict(X_test_combined)

In [ ]:
# Создание DataFrame для отправки
date = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
submit = pd.DataFrame({'has_next_round': y_pred })
submit.index = submit.index + 5512
submit.to_csv(f'submit_{date}.csv', index_label='index')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


In [ ]:
import random

seed = random.randint(1, 1000)
random.seed(seed)
print(seed)

In [ ]:
train_data = pd.read_csv('train.csv')


In [ ]:
# Преобразование даты
train_data['service_date'] = pd.to_datetime(train_data['service_date'])

# Извлечение дополнительных признаков
train_data['day'] = train_data['service_date'].dt.day
train_data['month'] = train_data['service_date'].dt.month
train_data['year'] = train_data['service_date'].dt.year
train_data['day_of_week'] = train_data['service_date'].dt.dayofweek

# Преобразование статуса заказа в числовые значения
status_mapping = {
    'Подтвержден': 1,
    'Аннулировано, без штрафа': 0,
    'Аннулировано, штраф': -1,
    # Добавьте дополнительные статусы, если есть
}
train_data['status_numeric'] = train_data['service_status'].map(status_mapping)

# Заполнение пропусков
train_data.fillna(0, inplace=True)


In [ ]:
features = ['day', 'month', 'year', 'day_of_week', 'hotel_id', 'hotel_category_star',
            'hotel_max_rooms', 'hotel_type', 'city_name', 'region_name',
            'country_name', 'status_numeric']
X = pd.get_dummies(train_data[features], drop_first=True)
y = train_data['sum_price']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

# Обучение модели
model = RandomForestRegressor(n_estimators=100, random_state=seed)
model.fit(X_train, y_train)

# Оценка модели на тестовой выборке
y_pred = model.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


In [ ]:
date_range = pd.date_range(start='2024-06-01', end='2024-06-30')
future_data = pd.DataFrame({'service_date': date_range})
future_data['day'] = future_data['service_date'].dt.day
future_data['month'] = future_data['service_date'].dt.month
future_data['year'] = future_data['service_date'].dt.year
future_data['day_of_week'] = future_data['service_date'].dt.dayofweek
# Здесь нужно добавить идентификатор отеля, звездность и другие поля
# Для примера допустим, что у нас есть несколько отелей с id от 1 до 5
future_data['hotel_id'] = np.random.randint(1, 6, size=len(future_data))
future_data['hotel_category_star'] = np.random.randint(1, 6, size=len(future_data))
future_data['hotel_max_rooms'] = np.random.randint(1, 101, size=len(future_data))
future_data['hotel_type'] = np.random.choice(['апартаменты', 'хостел', 'гостевой дом'], size=len(future_data))
future_data['city_name'] = 'Город1'
future_data['region_name'] = 'Регион1'
future_data['country_name'] = 'Страна1'
future_data['status_numeric'] = 1  # Предположим, что все заказы подтверждены

# Подготовим данные для предсказания
future_data_X = pd.get_dummies(future_data[features], drop_first=True)
predictions = model.predict(future_data_X)


In [ ]:
submission = pd.DataFrame({
    'service_date': future_data['service_date'],
    'predicted_sum_price': predictions
})
submission.to_csv(f'seed_{seed}.csv', index=False)


In [ ]:
!pip install dask-ml
!pip install dask-distributed


INFO: pip is looking at multiple versions of dask-expr to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.9/241.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.3/237.3 kB 12.5 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement dask-distributed (from versions: none)
ERROR: No matching distribution found for dask-distributed


In [ ]:
!pip install dask[complete] dask-ml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.2 MB/s eta 0:00:00


In [ ]:
!pip install vaex

  Preparing metadata (setup.py) ... done
  Using cached jedi-0.19.1-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.3/516.3 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.4/341.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!unzip train.zip

Archive:  train.zip
  inflating: train.csv               


In [ ]:
import vaex
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import random
import os

# Установка случайного семени
seed = random.randint(1, 1000)
random.seed(seed)
print("Seed:", seed)

# Загрузка данных из CSV
train_data = pd.read_csv('train.csv')

# Преобразование даты
train_data['service_date'] = pd.to_datetime(train_data['service_date'])

# Извлечение дополнительных признаков
train_data['day'] = train_data['service_date'].dt.day
train_data['month'] = train_data['service_date'].dt.month
train_data['year'] = train_data['service_date'].dt.year
train_data['day_of_week'] = train_data['service_date'].dt.dayofweek

# Преобразование статуса заказа в числовые значения
train_data['status_numeric'] = train_data['service_status'].map({
    'Подтвержден': 1,
    'Аннулировано, без штрафа': 0,
    'Аннулировано, штраф': -1
}).fillna(-2)

# Заполнение пропусков
train_data.fillna({'hotel_id': 0, 'hotel_category_star': 0, 'hotel_max_rooms': 0}, inplace=True)

# Сохранение в HDF5
hdf5_file = 'train_data.h5'
train_data.to_hdf(hdf5_file, key='data', mode='w')

# Загрузка данных из HDF5 с использованием Vaex
train_data = vaex.from_pandas(pd.read_hdf(hdf5_file, key='data'))

# Выбор признаков
features = ['day', 'month', 'year', 'day_of_week', 'hotel_id',
            'hotel_category_star', 'hotel_max_rooms', 'status_numeric']

# Преобразование категориальных переменных и формирование Vaex DataFrame
X_pandas = pd.get_dummies(train_data[features].to_pandas_df(), drop_first=True)

# Для извлечения целевого признака корректно
y = train_data['sum_price'].to_numpy()  # Используем to_numpy() вместо to_pandas()

# Разделение данных на тренировочные и тестовые
X_train, X_test, y_train, y_test = train_test_split(X_pandas, y, test_size=0.2, random_state=seed)

# Обучение модели
model = RandomForestRegressor(n_estimators=100, random_state=seed)
model.fit(X_train, y_train)

# Оценка модели на тестовой выборке
y_pred = model.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# Подготовка данных для будущего предсказания
date_range = pd.date_range(start='2024-06-01', end='2024-06-30')
future_data = pd.DataFrame({'service_date': date_range})
future_data['day'] = future_data['service_date'].dt.day
future_data['month'] = future_data['service_date'].dt.month
future_data['year'] = future_data['service_date'].dt.year
future_data['day_of_week'] = future_data['service_date'].dt.dayofweek
future_data['hotel_id'] = np.random.randint(1, 6, size=len(future_data))
future_data['hotel_category_star'] = np.random.randint(1, 6, size=len(future_data))
future_data['hotel_max_rooms'] = np.random.randint(1, 101, size=len(future_data))
future_data['status_numeric'] = 1  # Предположим, что все заказы подтверждены

# Подготовка данных для предсказания
future_data_X = pd.get_dummies(future_data[features], drop_first=True)

# Предсказание
predictions = model.predict(future_data_X)

# Создание датафрейма с предсказаниями
submission = pd.DataFrame({
    'service_date': future_data['service_date'],
    'predicted_sum_price': predictions
})
submission.to_csv(f'seed_{seed}.csv', index=False)

# Очистка временного файла
if os.path.exists(hdf5_file):
    os.remove(hdf5_file)


Seed: 322


<ipython-input-2-fb409b92197a>:39: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block4_values] [items->Index(['service_status', 'hotel_type', 'city_name', 'region_name',
       'country_name'],
      dtype='object')]

  train_data.to_hdf(hdf5_file, key='data', mode='w')


KeyboardInterrupt: 

In [ ]:
import vaex
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import random
import os

# Установка случайного семени
seed = random.randint(1, 1000)
random.seed(seed)
print("Seed:", seed)

# Загрузка данных из CSV
train_data = pd.read_csv('train.csv')

# Преобразование даты
train_data['service_date'] = pd.to_datetime(train_data['service_date'])

# Извлечение дополнительных признаков
train_data['day'] = train_data['service_date'].dt.day
train_data['month'] = train_data['service_date'].dt.month
train_data['year'] = train_data['service_date'].dt.year
train_data['day_of_week'] = train_data['service_date'].dt.dayofweek

# Преобразование статуса заказа в числовые значения
train_data['status_numeric'] = train_data['service_status'].map({
    'Подтвержден': 1,
    'Аннулировано, без штрафа': 0,
    'Аннулировано, штраф': -1
}).fillna(-2)

# Заполнение пропусков
train_data.fillna({'hotel_id': 0, 'hotel_category_star': 0, 'hotel_max_rooms': 0}, inplace=True)

# Сохранение в HDF5
hdf5_file = 'train_data.h5'
train_data.to_hdf(hdf5_file, key='data', mode='w')

# Загрузка данных из HDF5 с использованием Vaex
train_data = vaex.from_pandas(pd.read_hdf(hdf5_file, key='data'))

# Выбор признаков
features = ['day', 'month', 'year', 'day_of_week', 'hotel_id',
            'hotel_category_star', 'hotel_max_rooms', 'status_numeric']

# Преобразование категориальных переменных и формирование Vaex DataFrame
X = train_data[features]

# Для извлечения целевого признака
y = train_data['sum_price']

# Преобразование Vaex DataFrame в pandas для обучения модели
X_df = X.to_pandas_df()
y_df = y.to_numpy()  # Преобразуем в pandas Series

# Разделение данных на тренировочные и тестовые
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.2, random_state=seed)

# Обучение модели
model = HistGradientBoostingRegressor(
    max_iter=1000,
    random_state=seed,
    max_depth=5

)
model.fit(X_train, y_train)

# Оценка модели на тестовой выборке
y_pred = model.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# Подготовка данных для будущего предсказания
date_range = pd.date_range(start='2024-06-01', end='2024-06-30')
future_data = pd.DataFrame({'service_date': date_range})
future_data['day'] = future_data['service_date'].dt.day
future_data['month'] = future_data['service_date'].dt.month
future_data['year'] = future_data['service_date'].dt.year
future_data['day_of_week'] = future_data['service_date'].dt.dayofweek
future_data['hotel_id'] = np.random.randint(1, 6, size=len(future_data))
future_data['hotel_category_star'] = np.random.randint(1, 6, size=len(future_data))
future_data['hotel_max_rooms'] = np.random.randint(1, 101, size=len(future_data))
future_data['status_numeric'] = 1  # Предположим, что все заказы подтверждены

# Подготовка данных для предсказания
future_data_X = pd.get_dummies(future_data[features], drop_first=True)

# Предсказание
predictions = model.predict(future_data_X)

# Создание датафрейма с предсказаниями
submission = pd.DataFrame({
    'date': future_data['service_date'],
    'forecast_value': predictions
})
submission.to_csv(f'seed_{seed}.csv', index=False)

# Очистка временного файла
if os.path.exists(hdf5_file):
    os.remove(hdf5_file)


Seed: 775


<ipython-input-3-e90afa2c89b5>:39: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block4_values] [items->Index(['service_status', 'hotel_type', 'city_name', 'region_name',
       'country_name'],
      dtype='object')]

  train_data.to_hdf(hdf5_file, key='data', mode='w')


RMSE: 22167.75719273768


In [ ]:
# Создание датафрейма с предсказаниями
submission = pd.DataFrame({
    'date': future_data['service_date'],
    'forecast_value': predictions
})
submission.to_csv(f'seed_{seed}.csv', index=False)